Run the following before cell excecution: 

-`!pip install ollama`
- `ollama serve`
- `ollama pull gemma3:12b` 

In [1]:
!pip install ollama

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
anaconda-cloud-auth 0.1.4 requires pydantic<2.0, but you have pydantic 2.12.5 which is incompatible.



     ---------------------------------------- 0.0/90.6 kB ? eta -:--:--
     --------------------------- ------------ 61.4/90.6 kB 1.7 MB/s eta 0:00:01
     ---------------------------------------- 90.6/90.6 kB 1.7 MB/s eta 0:00:00
   ---------------------------------------- 0.0/73.5 kB ? eta -:--:--
   ---------------------------------------- 73.5/73.5 kB 2.0 MB/s eta 0:00:00
   ---------------------------------------- 0.0/78.8 kB ? eta -:--:--
   ---------------------------------------- 78.8/78.8 kB 4.3 MB/s eta 0:00:00
   ---------------------------------------- 0.0/463.6 kB ? eta -:--:--
   --------------- ------------------------ 174.1/463.6 kB 5.3 MB/s eta 0:00:01
   ---------------------------------------  460.8/463.6 kB 5.8 MB/s eta 0:00:01
   ---------------------------------------- 463.6/463.6 kB 4.9 MB/s eta 0:00:00
   ---------------------------------------- 0.0/2.0 MB ? eta -:--:--
   ------- -------------------------------- 0.4/2.0 MB 8.3 MB/s eta 0:00:01
   ------------

In [7]:
MODEL = "gemma3:4b" 


#change temperature and num_predict accordingly
def complete_document_sdk(prefix: str, *, model: str = MODEL, num_predict: int = 180, temperature: float = 0.5) -> str:
    import ollama

    response = ollama.generate(
        model=model,
        prompt=prefix,
        stream=False,
        raw=True,
        options={
            "num_predict": num_predict,
            "temperature": temperature,
        },
    )
    return response["response"]


In [8]:
doc_prefix_sdk = """Question: What is the capital of France?

Answer: """

completion_sdk = complete_document_sdk(doc_prefix_sdk)
print(doc_prefix_sdk + completion_sdk)


Question: What is the capital of France?

Answer: 
Paris is the capital of France.


In [9]:
completion_sdk

'\nParis is the capital of France.'

# Conversation history managememnt

In [12]:
class ConversationManager:
    def __init__(self, system_prompt, document_type="structured"):
        """
        Args:
            system_prompt: The system instruction (becomes INTRODUCTION)
            document_type: "conversation", "markdown", or "structured"
        """
        self.history = []
        self.system_prompt = system_prompt
        self.document_type = document_type
    
    def add_user_message(self, message):
        self.history.append({"role": "user", "content": message})
    
    def add_assistant_message(self, message):
        self.history.append({"role": "assistant", "content": message})
    
    def _get_history_string(self):
        """Convert history to string for the prompt"""
        if not self.history:
            return ""
        parts = []
        for msg in self.history:
            role = "Student" if msg["role"] == "user" else "Assistant"
            parts.append(f"{role}: {msg['content']}")
        return "\n".join(parts)
    
    def assemble_prompt(self, context, question):
        """Assemble prompt based on document type"""
        
        history = self._get_history_string()
        
        if self.document_type == "conversation":
            # Advice Conversation archetype
            prompt = f"""{self.system_prompt}

{history}

Student: Here is the context:
{context}

Student: {question}
Assistant: Based on the context above, my answer is:"""
        
        elif self.document_type == "markdown":
            # Markdown Report archetype
            prompt = f"""# HKBU Research Assistant

## System
{self.system_prompt}

## Previous Conversation
{history}

## Context
{context}

## Question
{question}

## Answer
Based on the context above, my answer is:
"""
        
        else:  # structured (default)
            # Structured Document archetype
            prompt = f"""<prompt>
  <system>{self.system_prompt}</system>
  <history>{history}</history>
  <context>{context}</context>
  <question>{question}</question>
</prompt>

<response>
Based on the context above, my answer is:"""
        
        return prompt
    

    MODEL = "gemma3:4b" 



    def get_response(self, user_message, context, temperature=0.3, max_tokens=300):
        import ollama
        import re
        self.add_user_message(user_message)
        prompt = self.assemble_prompt(context, user_message)
        
        response = ollama.generate(
            model='gemma3:4b',
            prompt=prompt,
            options={"temperature": temperature, "num_predict": max_tokens},
            raw=True
        )
            # Clean the response by removing any XML tags like </response>
        raw_response = response["response"]
        
        # Remove </response> tag and anything after it (if it appears)
        if '</response>' in raw_response:
            raw_response = raw_response.split('</response>')[0]
        
        # Also remove any other potential XML tags at the end
        raw_response = re.sub(r'</?\w+>\s*$', '', raw_response)
        
        answer = raw_response
        self.add_assistant_message(answer)
        return answer
    
    def clear_history(self):
        """Clear conversation history but KEEP the system prompt"""
        self.history = []
    
    def update_system_prompt(self, new_system_prompt):
        """Update system prompt mid-conversation"""
        self.system_prompt = new_system_prompt
    
    def set_document_type(self, document_type):
        """Change document archetype"""
        if document_type in ["conversation", "markdown", "structured"]:
            self.document_type = document_type
    
    def get_history(self):
        """Return full conversation history"""
        return self.history
    
    def display_conversation(self):
        """Display the full conversation history"""
        print("\n" + "="*50)
        print(f"Document Type: {self.document_type}")
        if self.system_prompt:
            print(f"System: {self.system_prompt}\n")
        for msg in self.history:
            role = "You" if msg['role'] == 'user' else "Assistant"
            print(f"{role}: {msg['content']}")
        print("="*50)

In [14]:
# Simple Terminal-Based Research Assistant
# Adjust these parameters at the beginning

# ========== CONFIGURATION VARIABLES ==========
SYSTEM_PROMPT = "You are an HKBU research assistant. Answer short."
DOCUMENT_TYPE = "structured"  # Options: 'structured', 'markdown', 'conversation'
TEMPERATURE = 0.3
MAX_TOKENS = 300
# ============================================

# Create ConversationManager
conversation = ConversationManager(
    system_prompt=SYSTEM_PROMPT,
    document_type=DOCUMENT_TYPE
)

print("="*60)
print("🤖 HKBU RESEARCH ASSISTANT")
print("="*60)
print("Type 'exit' to quit")
print(f"System: {SYSTEM_PROMPT}")
print("-"*60)

# Main conversation loop
user_input = ""
while user_input.lower() != 'exit':
    # Get user input
    user_input = input("\nType your message: ")
    
    if user_input.lower() == 'exit':
        print("\n👋 Goodbye!")
        break
    
    if not user_input.strip():
        continue
    
    # Print user message clearly
    print(f"\n👤 USER MESSAGE: {user_input}")
    
    # Context (you can modify this or make it dynamic)
    context = """"""
    
    # Get response with configured parameters
    response = conversation.get_response(
        user_message=user_input,
        context=context,
        temperature=TEMPERATURE,
        max_tokens=MAX_TOKENS
    )
    
    # Print assistant response clearly
    print(f"🤖 ASSISTANT RESPONSE: {response}")
    print("-"*40)  # Separator for readability
    

🤖 HKBU RESEARCH ASSISTANT
Type 'exit' to quit
System: You are an HKBU research assistant. Answer short.
------------------------------------------------------------

👤 USER MESSAGE: hi, my name is bob
🤖 ASSISTANT RESPONSE:  Hello Bob!

----------------------------------------

👤 USER MESSAGE: how are u
🤖 ASSISTANT RESPONSE:  I am doing well, thank you for asking!

----------------------------------------

👤 USER MESSAGE: whats my name
🤖 ASSISTANT RESPONSE:  Bob.

----------------------------------------

👋 Goodbye!


In [ ]:
# Create assistant with system prompt
system_prompt_concise = """You are an HKBU research assistant. 
Provide answers in 2-3 sentences maximum. Be direct and factual.
Always cite sources. Never add extra commentary."""

# Context
context = """The user is asking questions about various topics. 
Provide helpful, accurate responses based on available information."""

MODEL = "gemma3:4b" 

conversation = ConversationManager(system_prompt=system_prompt_concise)

print("🤖 Research Assistant Chat (type 'exit' to quit)")
print("-" * 50)

user_message = input("\n👤 You: ")

while user_message.lower() not in ['exit', 'quit']:
    response = conversation.get_response(user_message, context)
    print(f"🤖 Assistant: {response}")
    user_message = input("\n👤 You: ")

print("\n🤖 Assistant: Conversation ended. Goodbye!")
conversation.display_conversation()

🤖 Research Assistant Chat (type 'exit' to quit)
--------------------------------------------------
🤖 Assistant: Hello there! How can I assist you today? [No source needed]
</response>
🤖 Assistant: I'm sorry, I do not understand your question. Please rephrase it or provide more context. [No source needed]
</response>
🤖 Assistant: I am ready to assist you with your questions. Please provide your question so I can respond accurately. [No source needed]
</response>


In [5]:
import ipywidgets as widgets
from IPython.display import display

# Create ConversationManager FIRST
conversation = ConversationManager(
    system_prompt="You are an HKBU research assistant. Answer concisely with citations.",
    document_type="structured"
)

# Create UI components
output_area = widgets.HTML(value="<i>Start a conversation with your research assistant!</i>")
text_input = widgets.Text(placeholder="Type your message...", layout=widgets.Layout(width='60%'))
send_button = widgets.Button(description="Send", button_style='primary')
clear_button = widgets.Button(description="Clear Chat", button_style='warning')
settings_button = widgets.Button(description="⚙️ Settings", button_style='info')

# Settings panel (collapsible) - ALL settings included
settings_panel = widgets.Accordion([widgets.VBox([
    widgets.FloatSlider(value=0.3, min=0, max=1.0, step=0.1, description='Temperature:'),
    widgets.IntSlider(value=300, min=50, max=1000, step=50, description='Max tokens:'),
    widgets.Dropdown(
        options=['structured', 'markdown', 'conversation'],
        value='structured',
        description='Document Type:',
        style={'description_width': 'initial'}
    ),
    widgets.Textarea(
        value="You are an HKBU research assistant. Answer concisely with citations.", 
        description='System Prompt:', 
        layout=widgets.Layout(width='100%', height='100px')
    )
])])
settings_panel.set_title(0, 'Settings')

def on_send(b):
    question = text_input.value
    if not question:
        return
    text_input.value = ""
    
    # Get ALL current settings from UI
    current_temp = settings_panel.children[0].children[0].value
    current_max_tokens = settings_panel.children[0].children[1].value
    new_doc_type = settings_panel.children[0].children[2].value
    new_system_prompt = settings_panel.children[0].children[3].value
    
    # Update conversation settings if changed
    if new_system_prompt != conversation.system_prompt:
        conversation.update_system_prompt(new_system_prompt)
    
    if new_doc_type != conversation.document_type:
        conversation.set_document_type(new_doc_type)
    
    # Display user message
    output_area.value += f"""
    <div style='text-align:right; color: #1565C0; background:#E3F2FD; 
                padding:10px; margin:5px; border-radius:10px'>
        <b>👤 You:</b> {question}
    </div>
    """
    
    try:
        # TODO: Replace with your actual RAG context retrieval
        context = """[HKBU Calendar 2025-26] Semester 2: January 15 - May 15
[HKBU Library Hours] Weekdays: 8am-11pm, Saturday: 9am-9pm, Sunday: 1pm-9pm"""
        
        # Get response with ALL settings
        response = conversation.get_response(
            user_message=question,
            context=context,
            temperature=current_temp,
            max_tokens=current_max_tokens
        )
        
        # Display assistant response
        output_area.value += f"""
        <div style='text-align:left; color: #2E7D32; background:#E8F5E9; 
                    padding:10px; margin:5px; border-radius:10px'>
            <b>🤖 Assistant:</b> {response}
        </div>
        """
        
    except Exception as e:
        output_area.value += f"""
        <div style='color: red; background:#FFEBEE; padding:10px; margin:5px; border-radius:10px'>
            <b>❌ Error:</b> {str(e)}
        </div>
        """

def on_clear(b):
    conversation.clear_history()
    output_area.value = "<i>✅ Conversation cleared! Start a new chat below.</i>"

def on_settings(b):
    if settings_panel.layout.display == 'none':
        settings_panel.layout.display = 'block'
    else:
        settings_panel.layout.display = 'none'

# Connect buttons
send_button.on_click(on_send)
clear_button.on_click(on_clear)
settings_button.on_click(on_settings)

# Initially hide settings panel
settings_panel.layout.display = 'none'

# Display UI
print("="*60)
print("🤖 HKBU RESEARCH ASSISTANT")
print("="*60)
print("Ask questions about courses, schedules, policies, or study plans!")
print(f"📄 Current document type: {conversation.document_type}")
print("-"*60)

# Arrange layout
input_row = widgets.HBox([text_input, send_button, clear_button, settings_button])
display(settings_panel, output_area, input_row)

🤖 HKBU RESEARCH ASSISTANT
Ask questions about courses, schedules, policies, or study plans!
📄 Current document type: structured
------------------------------------------------------------


Accordion(children=(VBox(children=(FloatSlider(value=0.3, description='Temperature:', max=1.0), IntSlider(valu…

HTML(value='<i>Start a conversation with your research assistant!</i>')

## CLeaning up

In [2]:
class ConversationManager:
    def __init__(self, system_prompt, document_format="structured"):
        """
        Args:
            system_prompt: The system instruction (becomes INTRODUCTION)
            document_format: "conversation", "markdown", or "structured"
        """
        self.history = []
        self.system_prompt = system_prompt
        self.document_format = document_format
    
    def add_user_message(self, message):
        self.history.append({"role": "user", "content": message})
    
    def add_assistant_message(self, message):
        self.history.append({"role": "assistant", "content": message})
    
    def _get_history_string(self):
        """Convert history to string for the prompt"""
        if not self.history:
            return ""
        parts = []
        for msg in self.history:
            role = "Student" if msg["role"] == "user" else "Assistant"
            parts.append(f"{role}: {msg['content']}")
        return "\n".join(parts)
    
    def assemble_prompt(self, context, question):
        """Assemble prompt based on document format"""
        
        history = self._get_history_string()
        
        if self.document_format == "conversation":
            # Advice Conversation archetype
            prompt = f"""{self.system_prompt}

{history}

Student: Here is the context:
{context}

Student: {question}
Assistant: Based on the context above, my answer is:"""
        
        elif self.document_format == "markdown":
            # Markdown Report archetype
            prompt = f"""# HKBU Research Assistant

## System
{self.system_prompt}

## Previous Conversation
{history}

## Context
{context}

## Question
{question}

## Answer
Based on the context above, my answer is:
"""
        
        else:  # structured (default)
            # Structured Document archetype
            prompt = f"""<prompt>
  <system>{self.system_prompt}</system>
  <history>{history}</history>
  <context>{context}</context>
  <question>{question}</question>
</prompt>

<response>
Based on the context above, my answer is:"""
        
        return prompt
    
    def clear_history(self):
        """Clear conversation history but KEEP the system prompt"""
        self.history = []
    
    def update_system_prompt(self, new_system_prompt):
        """Update system prompt mid-conversation"""
        self.system_prompt = new_system_prompt
    
    def set_document_format(self, document_format):
        """Change document archetype"""
        if document_format in ["conversation", "markdown", "structured"]:
            self.document_format = document_format
    
    def get_history(self):
        """Return full conversation history"""
        return self.history
    
    def display_conversation(self):
        """Display the full conversation history"""
        print("\n" + "="*50)
        print(f"Document Format: {self.document_format}")
        if self.system_prompt:
            print(f"System: {self.system_prompt}\n")
        for msg in self.history:
            role = "You" if msg['role'] == 'user' else "Assistant"
            print(f"{role}: {msg['content']}")
        print("="*50)

In [3]:
MODEL = "gemma3:4b" 

def complete_document_sdk(prefix: str, *, model: str = MODEL, num_predict: int = 180, temperature: float = 0.5) -> str:
    import ollama
    import re

    response = ollama.generate(
        model=model,
        prompt=prefix,
        stream=False,
        raw=True,
        options={
            "num_predict": num_predict,
            "temperature": temperature,
        },
    )
    # Clean the response by removing any XML tags like </response>
    raw_response = response["response"]
    
    # Remove </response> tag and anything after it (if it appears)
    if '</response>' in raw_response:
        raw_response = raw_response.split('</response>')[0]
    
    # Also remove any other potential XML tags at the end
    raw_response = re.sub(r'</?\w+>\s*$', '', raw_response)
    
    return raw_response.strip()

In [ ]:
# Adjustable variables
TEMPERATURE = 0.7
MAX_TOKENS = 300
MODEL_NAME = "gemma3:4b"
DOCUMENT_FORMAT = "structured"

# System prompt
SYSTEM_PROMPT = "You are a helpful research assistant. Provide accurate, concise answers based on the given context."

# Initialize conversation manager
conv_manager = ConversationManager(
    system_prompt=SYSTEM_PROMPT,
    document_format=DOCUMENT_FORMAT
)

# Context
context = """The user is asking questions about various topics. 
Provide helpful, accurate responses based on available information."""

# Simple working chat loop
print("🤖 Research Assistant Chat (type 'exit' to quit)")
print("-" * 50)
print(f"Settings: Temperature={TEMPERATURE}, Max Tokens={MAX_TOKENS}, Format={DOCUMENT_FORMAT}")
print("-" * 50)

# Get first message
user_message = input("\n👤 You: ")
print(f"👤 You: {user_message}")  # ADD THIS - print the user input

while user_message.lower() not in ['exit', 'quit']:
    # Get response
    conv_manager.add_user_message(user_message)
    prompt = conv_manager.assemble_prompt(context, user_message)
    
    response = complete_document_sdk(
        prefix=prompt,
        model=MODEL_NAME,
        num_predict=MAX_TOKENS,
        temperature=TEMPERATURE
    )
    
    print(f"🤖 Assistant: {response}")
    conv_manager.add_assistant_message(response)
    
    # Get next message
    user_message = input("\n👤 You: ")
    print(f"👤 You: {user_message}")  # ADD THIS - print the user input

print("\nGoodbye! 👋")

🤖 Research Assistant Chat (type 'exit' to quit)
--------------------------------------------------
Settings: Temperature=0.7, Max Tokens=300, Format=structured
--------------------------------------------------
👤 You: 
🤖 Assistant: Please ask your question! I'm ready to assist you with research.
👤 You: hh
🤖 Assistant: I'm ready to assist you with research. Please ask your question!
👤 You: hh
🤖 Assistant: I'm ready to assist you with research. Please ask your question!
👤 You: 
🤖 Assistant: I'm ready to assist you with research. Please ask your question!
👤 You: 
🤖 Assistant: I'm ready to assist you with research. Please ask your question!
👤 You: 


# try 3

In [22]:
import ollama

# Response generation function
DEFAULT_MODEL = "gemma3:4b"

def complete_document_sdk(prefix: str, *, model: str = DEFAULT_MODEL, num_predict: int = 180, temperature: float = 0.5) -> str:
    response = ollama.generate(
        model=model,
        prompt=prefix,
        stream=False,
        raw=True,
        options={
            "num_predict": num_predict,
            "temperature": temperature,
        },
    )
    return response["response"]

In [ ]:
class ConversationManager:
    def __init__(self, system_prompt, document_type):

        self.history = []
        self.system_prompt = system_prompt
        self.document_type = document_type
    
    def add_user_message(self, message):
        self.history.append({"role": "user", "content": message})
    
    def add_assistant_message(self, message):
        self.history.append({"role": "assistant", "content": message})
    
    #formats the chat history to a student and assistant roles for the prompt
    def _get_history_string(self):
        if not self.history:
            return ""
        parts = []
        for msg in self.history:
            role = "Student" if msg["role"] == "user" else "Assistant"
            parts.append(f"{role}: {msg['content']}")
        return "\n".join(parts)
    
    #Assembling the prompt based on document type
    def assemble_prompt(self, context, question): 
        history = self._get_history_string()
        
        #Conversational format including transition to final answer
        if self.document_type == "conversation":
            #System prompt is the introduction
            prompt = f"""{self.system_prompt} 
{history}

Student: Here is the context:
{context}

Student: {question}
Assistant: Based on the context above, my answer is:"""
        

        #Markdown format including transition to final answer
        elif self.document_type == "markdown":
            prompt = f"""# HKBU Research Assistant

## System
{self.system_prompt}

## Previous Conversation
{history}

## Context
{context}

## Question
{question}

## Answer
Based on the context above, my answer is:
"""
        #Structured document format including transition to final answer
        else:  
            prompt = f"""<prompt>
  <system>{self.system_prompt}</system>
  <history>{history}</history>
  <context>{context}</context>
  <question>{question}</question>
</prompt>

<response>
Based on the context above, my answer is:"""
        
        return prompt
    

    #Retrieving and fomatting complete_document_sdk output
    def get_response(self, user_message, context, model, temperature=0.3, max_tokens=300):
        import re
        #Add user message to the conversation and assemble the prompt
        self.add_user_message(user_message)
        prompt = self.assemble_prompt(context, user_message)
        

        raw_response = complete_document_sdk(
            prefix=prompt,
            model=model,
            num_predict=max_tokens,
            temperature=temperature
        )
        
        #Cleaning the response
        if '</response>' in raw_response:
            raw_response = raw_response.split('</response>')[0]
        raw_response = re.sub(r'</?\w+>\s*$', '', raw_response)
        
        answer = raw_response

        #Adding the models response to the conversation
        self.add_assistant_message(answer)
        return answer
    
    def clear_history(self):
        """Clear conversation history but KEEP the system prompt"""
        self.history = []
    
    def update_system_prompt(self, new_system_prompt):
        """Update system prompt mid-conversation"""
        self.system_prompt = new_system_prompt
    
    def set_document_type(self, document_type):
        """Change document archetype"""
        if document_type in ["conversation", "markdown", "structured"]:
            self.document_type = document_type
    
    def get_history(self):
        """Return full conversation history"""
        return self.history
    
    def display_conversation(self):
        """Display the full conversation history"""
        print("\n" + "="*50)
        print(f"Document Type: {self.document_type}")
        if self.system_prompt:
            print(f"System: {self.system_prompt}\n")
        for msg in self.history:
            role = "You" if msg['role'] == 'user' else "Assistant"
            print(f"{role}: {msg['content']}")
        print("="*50)



In [24]:
# ========== CONFIGURATION VARIABLES ==========
SYSTEM_PROMPT = "You are an HKBU research assistant. Answer short." #System prompt
CONTEXT = """""" # RAG Implementation will go here, might have to change it to be dynamic
DOCUMENT_TYPE = "structured"  # Options: 'structured', 'markdown', 'conversation'
MODEL = "gemma3:4b"  # Model configuration variable
TEMPERATURE = 0.3
MAX_TOKENS = 180
# ============================================

# Create ConversationManager
conversation = ConversationManager(
    system_prompt=SYSTEM_PROMPT,
    document_type=DOCUMENT_TYPE
)

print("="*60)
print("🤖 HKBU RESEARCH ASSISTANT")
print("="*60)
print("Type 'exit' to quit")
print(f"System: {SYSTEM_PROMPT}")
print(f"Model: {MODEL}")
print("-"*60)

# Main conversation loop
user_input = ""
while user_input.lower() != 'exit':
    # Get user input
    user_input = input("\nType your message: ")
    
    if user_input.lower() == 'exit':
        print("\n👋 Goodbye!")
        break
    
    if not user_input.strip():
        continue
    
    
    print(f"\n👤 USER MESSAGE: {user_input}")
    
    
    # Get model response with configured parameters (conversation manager handles history and prompt assembly)
    response = conversation.get_response(
        user_message=user_input,
        context=CONTEXT,
        model=MODEL,  
        temperature=TEMPERATURE,
        max_tokens=MAX_TOKENS
    )
    
    # Print assistant response clearly
    print(f"🤖 ASSISTANT RESPONSE: {response}")
    print("-"*40)  # Separator for readability

🤖 HKBU RESEARCH ASSISTANT
Type 'exit' to quit
System: You are an HKBU research assistant. Answer short.
Model: gemma3:4b
------------------------------------------------------------

👤 USER MESSAGE: hi
🤖 ASSISTANT RESPONSE:  Hello!

----------------------------------------

👤 USER MESSAGE: hh
🤖 ASSISTANT RESPONSE:  Hello!

----------------------------------------

👋 Goodbye!
